In [0]:
from pyspark.sql import functions as F

failures = []


def check(name, condition, details=""):
    if condition:
        print(f"PASS: {name}")
    else:
        message = name if not details else f"{name}: {details}"
        failures.append(message)
        print(f"FAIL: {message}")


def check_row_count(table_name, expected):
    actual = spark.table(table_name).count()
    check(
        f"{table_name} row count",
        actual == expected,
        f"expected {expected}, got {actual}",
    )


# Regresjonstall for dagens faste kildesnapshot.
expected_counts = {
    "clubdata.bronze.matches_raw": 23,
    "clubdata.bronze.geocoding_raw": 11,
    "clubdata.bronze.weather_sources_raw": 14,
    "clubdata.bronze.weather_observations_raw": 8,
    "clubdata.silver.matches": 21,
    "clubdata.silver.venues": 9,
    "clubdata.silver.weather_observations": 657,
    "clubdata.gold.match_insights": 21,
}

for table_name, expected in expected_counts.items():
    check_row_count(table_name, expected)


matches = spark.table("clubdata.silver.matches")
venues = spark.table("clubdata.silver.venues")
weather = spark.table("clubdata.silver.weather_observations")
gold = spark.table("clubdata.gold.match_insights")


check(
    "Silver match_id is unique",
    matches.count()
    == matches.select("match_id").distinct().count(),
)

check(
    "Silver venue_id is unique",
    venues.count()
    == venues.select("venue_id").distinct().count(),
)

check(
    "Silver weather series is unique",
    weather.count()
    == weather.select(
        "venue_id",
        "weather_station_id",
        "observed_at",
        "element",
    ).distinct().count(),
)

required_match_errors = matches.filter(
    F.col("match_id").isNull()
    | F.col("kickoff_at").isNull()
    | F.col("home_team_name").isNull()
    | F.col("away_team_name").isNull()
).count()

check(
    "Required Silver match fields are populated",
    required_match_errors == 0,
    f"{required_match_errors} invalid rows",
)

invalid_coordinates = venues.filter(
    (
        F.col("latitude").isNotNull()
        & ~F.col("latitude").between(-90, 90)
    )
    | (
        F.col("longitude").isNotNull()
        & ~F.col("longitude").between(-180, 180)
    )
).count()

check(
    "Venue coordinates are valid",
    invalid_coordinates == 0,
    f"{invalid_coordinates} invalid rows",
)

invalid_weather = weather.filter(
    F.col("observed_at").isNull()
    | F.col("element").isNull()
    | F.col("value").isNull()
).count()

check(
    "Required weather fields are populated",
    invalid_weather == 0,
    f"{invalid_weather} invalid rows",
)

gold_ids = gold.select("match_id")
silver_ids = matches.select("match_id")

id_mismatches = (
    gold_ids.subtract(silver_ids)
    .unionByName(silver_ids.subtract(gold_ids))
    .count()
)

check(
    "Gold contains exactly the Silver matches",
    id_mismatches == 0,
    f"{id_mismatches} mismatched IDs",
)

invalid_results = gold.filter(
    F.col("result").isNotNull()
    & ~F.col("result").isin("win", "draw", "loss")
).count()

check(
    "Gold result values are valid",
    invalid_results == 0,
    f"{invalid_results} invalid results",
)

invalid_weather_offsets = gold.filter(
    F.col("weather_observed_at").isNotNull()
    & (
        F.abs(
            F.col("weather_observed_at").cast("long")
            - F.col("kickoff_at").cast("long")
        )
        > 3 * 60 * 60
    )
).count()

check(
    "Gold weather is within three hours of kickoff",
    invalid_weather_offsets == 0,
    f"{invalid_weather_offsets} invalid snapshots",
)

matches_with_weather = gold.filter(
    F.col("weather_observed_at").isNotNull()
).count()

matches_with_coordinates = gold.filter(
    F.col("latitude").isNotNull()
    & F.col("longitude").isNotNull()
).count()

check(
    "Expected weather coverage",
    matches_with_weather == 8,
    f"expected 8, got {matches_with_weather}",
)

check(
    "Expected coordinate coverage",
    matches_with_coordinates == 14,
    f"expected 14, got {matches_with_coordinates}",
)

result_distribution = {
    row["result"]: row["count"]
    for row in gold.groupBy("result").count().collect()
}

check(
    "Expected result distribution",
    result_distribution == {
        "win": 8,
        "draw": 5,
        "loss": 8,
    },
    str(result_distribution),
)


if failures:
    raise AssertionError(
        "Pipeline validation failed:\n- " + "\n- ".join(failures)
    )

print("\nAll ClubData pipeline checks passed.")